# Multi-region Transformer hyperparameter grid search

Random search over Transformer hyperparameters on the multi-region training data (CH, NOR, ISL, FR). The best configuration is `PARAMS_TRANSFORMER` in `scripts/config_models.py`, used by all other notebooks and listed in the paper's hyperparameter table.

**Steps**
1. Load the stake datasets and monthly inputs of the source and target regions, and split Central Asia, Alaska and Italy/Austria into sub-regions.
2. Split the source glaciers into training and validation glaciers. The validation glaciers are the test glaciers of the within-region splits of `2.3 Within_region` (about 16–24% of each region's measurements).
3. Sample 150 valid configurations from the parameter grid and (with `RUN_GS = True`) train one model per configuration, logging the validation loss and RMSE (pooled and per region).
4. Read the log and select the configuration with the lowest validation loss.

Datasets are cached in `cfg.dataPath/cache/TF_Europe/FD_MODEL/datasets/`, the grid-search log in `FD_MODEL/GS_results/pooled_tf_gs_2026-06-12.csv`. The functions used here live in `scripts/dataset/fd_model.py`, `scripts/models/transformer_grid_search.py` and `scripts/plotting/gridsearch.py`.

## Setting up:

In [ ]:
import os

# Silence KeOps (used by geomloss for Sinkhorn distances) before it is imported
os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

import torch

# Initialise CUDA before the other imports: changing this order can change the
# GPU results slightly (~1e-7)
if torch.cuda.is_available():
    torch.cuda.init()

import sys
import logging
import warnings
from pathlib import Path

import pandas as pd
import massbalancemachine as mbm

sys.path.append(os.path.join(os.getcwd(), '../../'))  # Add root of repo to import MBM
# Shared config (paths, MONTHLY_COLS, STATIC_COLS, VOIS_CLIMATE, paths_multi,
# IT_GLACIERS/AT_GLACIERS, ...), model settings (PARAMS_TRANSFORMER) and helpers
from papers.Transformer_TF.scripts.config_TF_Europe import *
from papers.Transformer_TF.scripts.config_models import *
from papers.Transformer_TF.scripts.dataset import *
from papers.Transformer_TF.scripts.plotting import *
from papers.Transformer_TF.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

# Call config file (data paths, seed, etc.)
cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)  # reproducibility
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

# Use the GPU if one is visible
device = torch.device("cuda" if torch.cuda.device_count() > 0 else "cpu")
print(f"Device: {device}")

# Cache folders: grid-search datasets (CACHE_DIR) and logs (GS_RESULTS_DIR)
BASE_DIR = Path(cfg.dataPath) / path_cache / "FD_MODEL"
CACHE_DIR = BASE_DIR / "datasets/"
GS_RESULTS_DIR = BASE_DIR / "GS_results"
os.makedirs(CACHE_DIR, exist_ok=True)
GS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Base directory for data: {BASE_DIR}")

## Data:
### Stakes and monthly inputs:
The target regions are loaded too, because the month padding of the datasets is derived from all source–target pairs (`prepare_xreg_pairs_from_cache`). With `run_flag_by_code = False`, the monthly data are loaded from `WGMS/CrossRegional/monthly_cache/`.

In [ ]:
# Stake datasets of all regions configured in RGI_REGIONS (one DataFrame per RGI region)
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

# Add the RGI O2 sub-region of each glacier in Central Asia and Alaska
dfs, glacier_dict_ca = add_o2region_to_dfs(dfs, '13', cfg)
dfs, glacier_dict_alaska = add_o2region_to_dfs(dfs,
                                               '01',
                                               cfg,
                                               deduplicate_glaciers=True)

SOURCE_REGIONS = ['CH', 'NOR', 'ISL', "FR"]
TARGET_REGIONS = ['IT_AT', 'SJM', 'CENTRALASIA', 'ALA']

run_flag_by_code = {code: False for code in SOURCE_REGIONS + TARGET_REGIONS}  # True = recompute

monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=VOIS_CLIMATE,
    vois_topographical=STATIC_COLS,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

### Sub-regions and source–target pairs:

In [ ]:
# O2 sub-region labels in the monthly data of Central Asia and Alaska
monthly_cache = add_o2region_to_monthly_cache(monthly_cache, 'CENTRALASIA',
                                              glacier_dict_ca)
monthly_cache = add_o2region_to_monthly_cache(monthly_cache, 'ALA',
                                              glacier_dict_alaska)

# Italy / Austria: split by country
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'IT_AT',
    subregions={
        'IT': IT_GLACIERS,
        'AT': AT_GLACIERS
    },
    drop_parent=True,
)

# Central Asia sub-regions
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'CENTRALASIA',
    subregions={
        'CA_12': {
            'o2region_col': 'O2Region',
            'values': ['1', '2']
        },
        'CA_3': {
            'o2region_col': 'O2Region',
            'values': ['3']
        },
        'CA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
    },
)

# Alaska sub-regions
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'ALA',
    subregions={
        'ALA_2': {
            'o2region_col': 'O2Region',
            'values': ['2']
        },
        'ALA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
        'ALA_6': {
            'o2region_col': 'O2Region',
            'values': ['6']
        },
    },
)

TARGET_REGIONS_SUB = [
    'IT', 'SJM', 'CENTRALASIA', 'ALA', 'CA_12', 'CA_3', 'CA_4', 'ALA_2',
    'ALA_4', 'ALA_6'
]

# One pair per source region: (source, all other regions)
XREG_PAIRS = [(src,
               [r for r in SOURCE_REGIONS + TARGET_REGIONS_SUB if r != src])
              for src in SOURCE_REGIONS]

# Source data and month padding per pair (from the monthly cache, no ERA5 reprocessing)
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

months_head_pad = res_xreg_by_source["CH"]['months_head_pad']
months_tail_pad = res_xreg_by_source["CH"]['months_tail_pad']
print(f"months_head_pad: {months_head_pad}")
print(f"months_tail_pad: {months_tail_pad}")

### Training and validation glaciers:
The validation glaciers of each source region are its within-region test glaciers (`2.3 Within_region`); all other source glaciers are used for training. The datasets are built once (or loaded from `CACHE_DIR`) and reused for all configurations.

In [ ]:
WITHIN_SPLITS_CACHE = Path(cfg.dataPath) / path_cache / "WITHIN_REGION" / "within_region_splits_cache.pkl"
val_glaciers_by_region = load_within_region_test_glaciers(WITHIN_SPLITS_CACHE,
                                                          SOURCE_REGIONS)

gs_assets = build_grid_search_assets(
    res_xreg_by_source=res_xreg_by_source,
    val_glaciers_by_region=val_glaciers_by_region,
    source_regions=SOURCE_REGIONS,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    cache_dir=CACHE_DIR,
    force_recompute=False,
)
ds_gs_train = gs_assets["ds_gs_train_assets"]["ds_train"]
val_datasets = gs_assets["val_datasets"]
gs_train_glaciers = gs_assets["gs_train_glaciers"]
gs_val_glaciers = gs_assets["gs_val_glaciers"]

## Grid search:
### Parameter grid:
Combinations where `nhead` does not divide `d_model`, or `dim_feedforward` < `d_model`, are skipped; 150 of the remaining combinations are sampled.

In [ ]:
# Fixed parameters: input dimensions and sequence length
const_params = {
    "Fm": ds_gs_train.Xm.shape[-1],  # 8 monthly inputs
    "Fs": ds_gs_train.Xs.shape[-1],  # 3 static inputs
    "two_heads": False,
    "loss_spec": None,
    "T_max": 32,
}

param_grid = {
    "d_model":         [64, 128, 256],
    "nhead":           [2, 4, 8],          # filtered: must divide d_model
    "num_layers":      [2, 3, 4],
    "dim_feedforward": [128, 256, 512],    # filtered: must be >= d_model
    "dropout":         [0.0, 0.1, 0.2],
    "static_layers":   [1, 2],
    "static_hidden":   [64, 128],
    "static_dropout":  [0.0, 0.1],
    "head_dropout":    [0.0, 0.05, 0.1],
    "lr":              [5e-4, 1e-3, 2e-3],
    "weight_decay":    [1e-5, 1e-4],
}

N_SAMPLES = GS_N_TRIALS  # 150, set in scripts/config_models.py
sampled_params = sample_transformer_param_sets(
    const_params, param_grid, n_samples=N_SAMPLES, seed=cfg.seed,
)
print(f"Running {len(sampled_params)} random configurations")

### Run:
Trains one model per configuration (up to 150 epochs, early stopping on the pooled validation glaciers). This takes many hours, so it only runs with `RUN_GS = True`, which overwrites the log below.

In [ ]:
RUN_GS = False
log_filename = GS_RESULTS_DIR / "pooled_tf_gs_2026-06-12.csv"

if RUN_GS:
    best_overall = run_pooled_transformer_grid_search(
        cfg,
        device,
        ds_gs_train,
        val_datasets,
        SOURCE_REGIONS,
        sampled_params,
        log_filename=log_filename,
        epochs=GS_MAX_EPOCHS,
    )

### Results:

In [ ]:
# All configurations, best (lowest validation loss) first
results_gs = pd.read_csv(log_filename).sort_values(by='valid_loss')
results_gs

In [ ]:
# Hyperparameter values of the 10 best configurations
fig = plot_gs_top_configs(results_gs, param_cols=GS_PARAM_COLS, k=10)

In [ ]:
# Best configuration, as a parameter dict; this is PARAMS_TRANSFORMER in scripts/config_models.py
PARAMS_TRANSFORMER_POOLED = best_params_from_gs_log(results_gs, verbose=False)

print("PARAMS_TRANSFORMER_POOLED = {")
for k, v in PARAMS_TRANSFORMER_POOLED.items():
    print(f'    "{k}": {repr(v)},')
print("}")
print("Same as PARAMS_TRANSFORMER:", PARAMS_TRANSFORMER_POOLED == PARAMS_TRANSFORMER)